In [16]:
import cv2 
import numpy as np 
import random 
import os 
def generate_synthetic_image(object_img, background_img): 
    """ 
    Генерирует синтетическое изображение путем наложения объекта на фон 
    Args: 
        object_img: изображение объекта с альфа-каналом (RGBA) 
        background_img: фоновое изображение (RGB) 
    Returns: 
        synthetic_img: синтетическое изображение 
    """ 
    # Берем размеры фотографии через функцию shape
    bg_h, bg_w, _ = background_img.shape
    obj_h, obj_w = object_img.shape[:2] #Берем высоту и ширину объекта

    #Берем допустимый масштаб - чтобы предотвратить выход за границы фона
    scale_w = bg_w / obj_w
    scale_h = bg_h / obj_h
    max_scale = min(scale_w, scale_h) * 0.9  # небольшой запас
    
    # Случайное масштабирование объекта (0.3-0.7 от размера фона) 
    scale = random.uniform(0.3, min(0.7, max_scale))

    new_w = int(obj_w * scale)
    new_h = int(obj_h * scale)
    object_resized = cv2.resize(object_img, (new_w, new_h))

    # Случайный поворот объекта (-30 до +30 градусов) 
    angle = random.randint(-30, 30) 
    center = (new_w // 2, new_h // 2)
    rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)
    object_rotated = cv2.warpAffine(object_resized,
         rot_mat, (new_w, new_h),
           borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))
    
    # Обновляем размеры ПОСЛЕ поворота - чтобы предотвратить выход за границы фона
    rot_h, rot_w = object_rotated.shape[:2]

    # Защита от выхода за границы
    if rot_w >= bg_w or rot_h >= bg_h:
        return background_img
    # Горизонтальное отражение с вероятностью 0.5
    if random.random() < 0.5:
        object_rotated = cv2.flip(object_rotated, 1)
    
    # Случайная позиция на фоне 
    max_x = bg_w - rot_w
    max_y = bg_h - rot_h    
    x = random.randint(0, max_x)
    y = random.randint(0, max_y)
    # Реализовать наложение объекта на фон 
    obj_rgb = object_rotated[:, :, :3]
    obj_alpha = object_rotated[:, :, 3] / 255.0
    obj_alpha = obj_alpha[:, :, np.newaxis]

    synthetic_img = background_img.copy()
    roi = synthetic_img[y:y+new_h, x:x+new_w]

    roi[:] = obj_alpha * obj_rgb + (1 - obj_alpha) * roi

    # Применить аугментации (изменение яркости, контраста, размытие) 
    alpha = random.uniform(0.8, 1.2)   # контраст
    beta = random.randint(-30, 30)     # яркость
    synthetic_img = cv2.convertScaleAbs(synthetic_img, alpha=alpha, beta=beta)

    # Небольшое размытие
    if random.random() < 0.3:
        synthetic_img = cv2.GaussianBlur(synthetic_img, (5, 5), 0)

    return synthetic_img

In [17]:
#Генерация искусственных изображений
def load_random_object(class_name):
    folder = class_name
    files = [f for f in os.listdir(folder) if f.lower().endswith(".png")]
    file = random.choice(files)
    path = os.path.join(folder, file)
    return cv2.imread(path, cv2.IMREAD_UNCHANGED)  # важно: альфа!

In [18]:
def load_random_background():
    folder = "background"
    files = [f for f in os.listdir(folder) if f.lower().endswith(".jpg")]
    file = random.choice(files)
    path = os.path.join(folder, file)
    return cv2.imread(path)

In [19]:
def save(img, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cv2.imwrite(path, img)

In [20]:
classes = ["Bike", "Car", "FighterJet", "Helicopter", "Tank"]

sizes = {"train": 600, "validation": 101, "test": 101}

for split in ["train", "validation", "test"]:
    N = sizes[split]
    for class_name in classes:
        for i in range(N):
            obj = load_random_object(class_name)
            bg = load_random_background()
            img = generate_synthetic_image(obj, bg)

            path = f"dataset/{split}/{class_name}/{i:04d}.jpg"
            save(img, path)

        print(f"{split}/{class_name} — готово")

train/Bike — готово
train/Car — готово
train/FighterJet — готово
train/Helicopter — готово
train/Tank — готово
validation/Bike — готово
validation/Car — готово
validation/FighterJet — готово
validation/Helicopter — готово
validation/Tank — готово
test/Bike — готово
test/Car — готово
test/FighterJet — готово
test/Helicopter — готово
test/Tank — готово
